Databricks notebook source
MAGIC %md
MAGIC
MAGIC ## UMAP for patients with paternal infertility vs vasectomy patients (but can include patients who have had vasectomy)
MAGIC
MAGIC Note: This notebook includes all diagnoses

COMMAND ----------

In [29]:
os.chdir('/Users/fengxie/Documents/Logistic_Regression_Python_Stanford/Logistic_Regression_Python_MI')
from MI_Functions import *

COMMAND ----------

In [30]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy.core.multiarray
import numpy as np
import os
import umap
import re
import scipy
from scipy import stats
from scipy.stats import mstats
from scikit_posthocs import posthoc_dunn
import matplotlib
import re

In [31]:
#s.getcwd()


COMMAND ----------

MAGIC %md
MAGIC ## 'Import' functions

COMMAND ----------

MAGIC %run MI_Functions.py

COMMAND ----------

MAGIC %md
MAGIC ## Read in demographics and diagnoses files for male infertility and vasectomy patients

COMMAND ----------

MAGIC %md
MAGIC ### Demographics

COMMAND ----------

In [32]:
demographics = dict()

In [33]:
demos = ['mi',
         'vas_only']
file_names_demos = ['mi_pts_only_final',
                    'vas_pts_only_final']

In [34]:
for demo, file_name_demo in zip(demos, file_names_demos):
  temp = pd.read_pickle(f"male_infertility_validation/demographics/{file_name_demo}.pkl")
  demographics[demo] = temp

COMMAND ----------

MAGIC %md
MAGIC ### Diagnoses

COMMAND ----------

In [35]:
diagnoses = dict()

In [36]:
diags = ['mi_diag', 
         'vas_only_diag']
file_names_diags = ['mi_phe', 
                    'vas_phe']

In [37]:
for diag, file_name_diag in zip(diags, file_names_diags):
  temp = pd.read_pickle(f"male_infertility_validation/diagnoses/{file_name_diag}.pkl")
  diagnoses[diag] = temp

COMMAND ----------

MAGIC %md
MAGIC ## Make pivot table containing all patients' phenotypic profiles

COMMAND ----------

In [38]:
pivot_tables = dict()

In [39]:
for file_name_diag in diagnoses:
  pivot_tables[file_name_diag] = make_pivot_tables(diagnoses[file_name_diag], file_name_diag, n='phenotype')

%%% Making pivot table for mi_diag %%%
Shape of pivot table: (5551, 1549)


phenotype,Abdominal aortic aneurysm,Abdominal hernia,Abdominal pain,Abnormal arterial blood gases,Abnormal chest sounds,Abnormal coagulation profile,Abnormal electrocardiogram [ECG] [EKG],Abnormal findings examination of lungs,Abnormal findings on exam of gastrointestinal tract/ abdominal area,Abnormal findings on study of brain and/or nervous system,...,Vitamin deficiency,Vitiligo,Voice disturbance,Von willebrand's disease,Wegener's granulomatosis,Wheezing,potential health hazards related to communicable diseases\npotential health hazards related to communicable diseases,severe protein-calorie malnutrition,"stress incontinence, female",male infertility status
person_id,,,,,,,,,,,,,,,,,,,,,
29939065,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
29939067,0,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
29939074,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1




Done


%%% Making pivot table for vas_only_diag %%%
Shape of pivot table: (2464, 1358)


phenotype,Abdominal aortic aneurysm,Abdominal hernia,Abdominal pain,Abnormal arterial blood gases,Abnormal coagulation profile,Abnormal electrocardiogram [ECG] [EKG],Abnormal findings examination of lungs,Abnormal findings on exam of gastrointestinal tract/ abdominal area,Abnormal findings on study of brain and/or nervous system,Abnormal function study of cardiovascular system,...,Vitamin B-complex deficiencies,Vitamin D deficiency,Vitamin deficiency,Vitiligo,Voice disturbance,Von willebrand's disease,Wheezing,potential health hazards related to communicable diseases\npotential health hazards related to communicable diseases,severe protein-calorie malnutrition,male infertility status
person_id,,,,,,,,,,,,,,,,,,,,,
29940002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
29940245,0,0,1,0,0,1,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
29940249,0,0,0,0,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0




Done




COMMAND ----------

MAGIC %md
MAGIC ### Concatenate pivot tables

COMMAND ----------

In [40]:
alldiag_pivot = pd.concat([pivot_tables['mi_diag'], pivot_tables['vas_only_diag']], axis=0)

COMMAND ----------

MAGIC %md
MAGIC #### Drop infertility-related conditions

COMMAND ----------

In [41]:
colstodrop = list()

In [42]:
colstodrop.extend(alldiag_pivot.columns[alldiag_pivot.columns.str.contains('infertility|spermia|reproduction',      
                                                                           flags=re.IGNORECASE)])

In [43]:
# remove male infertility status as a column to drop
colstodrop.remove('male infertility status')

In [44]:
colstodrop = set(colstodrop)
print(colstodrop)

{'Infertility, male', 'Infertility, female, associated with anovulation', 'Azoospermia and oligospermia', 'Persons encountering health services in circumstances related to reproduction', 'Infertility, female'}


COMMAND ----------

In [45]:
alldiag_pivot = alldiag_pivot.drop(colstodrop, axis=1)

COMMAND ----------

MAGIC %md
MAGIC ## Make demographic dfs

COMMAND ----------

In [46]:
demographics['mi'].columns

Index(['person_id', 'year_of_birth', 'estimated_age', 'gender_concept_id',
       'gender', 'race_concept_id', 'race', 'ethnicity_concept_id',
       'ethnicity', 'num_visits_total', 'emr_months_total',
       'first_mi_or_vas_date', 'analysis_cutoff_date', 'mi_or_vas_est_age',
       'num_visits_before', 'num_visits_same', 'num_visits_after',
       'emr_months_before', 'emr_months_after'],
      dtype='object')

COMMAND ----------

In [47]:
demographic_cols = ['person_id', 'year_of_birth', 'estimated_age', 'gender', 'race',
                    'ethnicity'] #'location_source_value', 'adi', 'adi_category']

COMMAND ----------

In [48]:
# Only keep demographic columns and merge
demographics['mi'] = demographics['mi'][demographic_cols].copy()
demographics['vas_only'] = demographics['vas_only'][demographic_cols].copy()

In [49]:
all_demo_df = pd.concat([demographics['mi'], demographics['vas_only']], axis=0, copy=False)

COMMAND ----------

In [50]:
all_demo_df = all_demo_df.drop_duplicates(). \
                          set_index('person_id').reindex(alldiag_pivot.index)

COMMAND ----------

MAGIC %md
MAGIC ## Check that alldiag_pivot and all_demo dfs have the same number of rows, the same index, and fillna with 0.

COMMAND ----------

MAGIC %md
MAGIC ### fillna with 0

COMMAND ----------

In [51]:
all_demo_df = all_demo_df.fillna(0)
alldiag_pivot = alldiag_pivot.fillna(0)

COMMAND ----------

MAGIC %md
MAGIC ### Check the shape of demo and diag dfs

COMMAND ----------

In [52]:
print(f"Shape of all_demo_df is {all_demo_df.shape}")
print(f"Shape of alldiag_pivot is {alldiag_pivot.shape}")
print(f"Number of rows of the dataframes are the same: {all_demo_df.shape[0] == alldiag_pivot.shape[0]}")
print('\n')

Shape of all_demo_df is (8015, 5)
Shape of alldiag_pivot is (8015, 1574)
Number of rows of the dataframes are the same: True




Shape of all_demo_df is (8015, 5)
Shape of alldiag_pivot is (8015, 1574)
Number of rows of the dataframes are the same: True

COMMAND ----------

MAGIC %md
MAGIC ### Check whether indices are the same for demo and diag dfs

COMMAND ----------

In [53]:
demo_index = all_demo_df.index
diag_index = alldiag_pivot.index
print(f"indices are the same for all_demo_df and alldiag_pivot: {demo_index.equals(diag_index)}")
print('\n')

indices are the same for all_demo_df and alldiag_pivot: True




COMMAND ----------

MAGIC %md
MAGIC ## Dimensionality Reduction

COMMAND ----------

In [54]:
X = dimensionality_reduction(diag=alldiag_pivot, file_name='mi_vas_only')

Peforming dimensionality reduction for mi_vas_only...
UMAP(angular_rp_forest=True, metric='cosine', random_state=42, verbose=True)
Fri Sep 15 02:22:14 2023 Construct fuzzy simplicial set
Fri Sep 15 02:22:14 2023 Finding Nearest Neighbors
Fri Sep 15 02:22:14 2023 Building RP forest with 9 trees
Fri Sep 15 02:22:16 2023 NN descent for 13 iterations
	 1  /  13
	 2  /  13
	 3  /  13
	 4  /  13
	 5  /  13
	 6  /  13
	 7  /  13
	 8  /  13
	Stopping threshold met -- exiting after 8 iterations
Fri Sep 15 02:22:20 2023 Finished Nearest Neighbor Search
Fri Sep 15 02:22:21 2023 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

Fri Sep 15 02:22:31 2023 Finished embedding


,index,0,1
8010,8010,2.866424,-12.691486
8011,8011,1.666085,-12.070016
8012,8012,-0.603681,-8.385965
8013,8013,7.126637,-10.329717
8014,8014,-0.032231,-11.895370


Saving X as pandas DataFrame...
Saved


COMMAND ----------

MAGIC %md
MAGIC ### Save 'y', which will preserve each patient's paternal infertility status and other demographic features (it is not preserved after performing dimensionality reduction)

COMMAND ----------

In [55]:
temp = alldiag_pivot.copy()
y = temp['male infertility status'].replace({1 : 'male infertility patient', 0 : 'control (vasectomy patient)'})
y = y.to_frame()
y = y.reset_index()
y = y.reset_index()
y = y.merge(all_demo_df, on='person_id')
y.to_pickle("male_infertility_validation/tables/umap/y_all.pkl")

COMMAND ----------

In [56]:
y['male infertility status'].value_counts()

male infertility patient       5551
control (vasectomy patient)    2464
Name: male infertility status, dtype: int64

COMMAND ----------

MAGIC %md
MAGIC ## Save demographic dataframes

COMMAND ----------

In [57]:
temp = all_demo_df.copy()
temp = temp.reset_index()
temp.to_pickle("male_infertility_validation/tables/umap/demo_all.pkl")

In [58]:
print(f"Shape of alldiag_pivot is {alldiag_pivot.shape}")

Shape of alldiag_pivot is (8015, 1574)


COMMAND ----------